<a href="https://colab.research.google.com/github/varshini-cit/DAA-LAB/blob/main/2A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import random
import string
import time
import csv


# ============================================================
# Comparative Analysis of String Matching Algorithms
# Naive, Rabin-Karp, and KMP
# ============================================================

TEXT_LENGTH = 10000
PATTERN_LENGTHS = [5, 10, 20, 50]

# Fixed seed so the experiment is reproducible
random.seed(42)


# ------------------------------------------------------------
# Generate Text
# ------------------------------------------------------------
def generate_text(length):
    characters = string.ascii_lowercase
    return ''.join(random.choice(characters) for _ in range(length))


# ------------------------------------------------------------
# Generate Pattern
# ------------------------------------------------------------
def generate_pattern(text, length):
    # Select a random substring from the text.
    # This guarantees that the pattern occurs at least once.
    start = random.randint(0, len(text) - length)
    return text[start:start + length]


# ------------------------------------------------------------
# 1. Naive String Matching
# ------------------------------------------------------------
def naive_search(text, pattern):
    n = len(text)
    m = len(pattern)

    comparisons = 0
    matches = []

    for i in range(n - m + 1):
        j = 0

        while j < m:
            comparisons += 1

            if text[i + j] != pattern[j]:
                break

            j += 1

        if j == m:
            matches.append(i)

    return matches, comparisons


# ------------------------------------------------------------
# 2. Rabin-Karp Algorithm
# ------------------------------------------------------------
def rabin_karp_search(text, pattern):
    n = len(text)
    m = len(pattern)

    if m > n:
        return [], 0

    # Hash parameters
    base = 256
    prime = 101

    comparisons = 0
    matches = []

    pattern_hash = 0
    text_hash = 0
    h = 1

    # h = pow(base, m-1) % prime
    for _ in range(m - 1):
        h = (h * base) % prime

    # Calculate initial hash values
    for i in range(m):
        pattern_hash = (base * pattern_hash + ord(pattern[i])) % prime
        text_hash = (base * text_hash + ord(text[i])) % prime

    # Slide the pattern over the text
    for i in range(n - m + 1):

        # Hash values match
        if pattern_hash == text_hash:

            j = 0

            while j < m:
                comparisons += 1

                if text[i + j] != pattern[j]:
                    break

                j += 1

            if j == m:
                matches.append(i)

        # Calculate hash for next window
        if i < n - m:
            text_hash = (
                base * (text_hash - ord(text[i]) * h)
                + ord(text[i + m])
            ) % prime

            if text_hash < 0:
                text_hash += prime

    return matches, comparisons


# ------------------------------------------------------------
# 3. KMP Algorithm
# ------------------------------------------------------------
def build_lps(pattern):
    """
    Build the Longest Prefix Suffix (LPS) array.
    """

    m = len(pattern)
    lps = [0] * m

    length = 0
    i = 1

    while i < m:

        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1

        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1

    return lps


def kmp_search(text, pattern):
    n = len(text)
    m = len(pattern)

    lps = build_lps(pattern)

    comparisons = 0
    matches = []

    i = 0
    j = 0

    while i < n:

        comparisons += 1

        if text[i] == pattern[j]:
            i += 1
            j += 1

            if j == m:
                matches.append(i - j)
                j = lps[j - 1]

        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    return matches, comparisons


# ------------------------------------------------------------
# Measure Execution Time
# ------------------------------------------------------------
def measure_algorithm(algorithm, text, pattern):

    start_time = time.perf_counter()

    matches, comparisons = algorithm(text, pattern)

    end_time = time.perf_counter()

    execution_time = (end_time - start_time) * 1000

    return matches, comparisons, execution_time


# ------------------------------------------------------------
# Main Experiment
# ------------------------------------------------------------
def main():

    print("=" * 80)
    print("COMPARATIVE ANALYSIS OF STRING MATCHING ALGORITHMS")
    print("=" * 80)

    print(f"\nText Length: {TEXT_LENGTH}")
    print(f"Pattern Lengths: {PATTERN_LENGTHS}")

    # Generate one common text
    text = generate_text(TEXT_LENGTH)

    algorithms = {
        "Naive": naive_search,
        "Rabin-Karp": rabin_karp_search,
        "KMP": kmp_search
    }

    results = []

    print("\n" + "-" * 80)

    for pattern_length in PATTERN_LENGTHS:

        pattern = generate_pattern(text, pattern_length)

        print(f"\nPattern Length: {pattern_length}")
        print(f"Pattern: {pattern}")

        print("\nAlgorithm       Comparisons       Time (ms)       Matches")
        print("-" * 65)

        for name, algorithm in algorithms.items():

            matches, comparisons, execution_time = measure_algorithm(
                algorithm,
                text,
                pattern
            )

            print(
                f"{name:<15}"
                f"{comparisons:<18}"
                f"{execution_time:<16.4f}"
                f"{len(matches)}"
            )

            results.append({
                "Pattern Length": pattern_length,
                "Algorithm": name,
                "Character Comparisons": comparisons,
                "Execution Time (ms)": round(execution_time, 4),
                "Number of Matches": len(matches)
            })

    # --------------------------------------------------------
    # Save Results to CSV
    # --------------------------------------------------------
    with open("results.csv", "w", newline="") as file:

        fieldnames = [
            "Pattern Length",
            "Algorithm",
            "Character Comparisons",
            "Execution Time (ms)",
            "Number of Matches"
        ]

        writer = csv.DictWriter(file, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(results)

    print("\n" + "=" * 80)
    print("Experiment completed successfully.")
    print("Results saved to: results.csv")
    print("=" * 80)


# ------------------------------------------------------------
# Program Entry Point
# ------------------------------------------------------------
if __name__ == "__main__":
    main()

COMPARATIVE ANALYSIS OF STRING MATCHING ALGORITHMS

Text Length: 10000
Pattern Lengths: [5, 10, 20, 50]

--------------------------------------------------------------------------------

Pattern Length: 5
Pattern: rwufr

Algorithm       Comparisons       Time (ms)       Matches
-----------------------------------------------------------------
Naive          10431             1.8479          1
Rabin-Karp     114               3.0166          1
KMP            10414             1.3229          1

Pattern Length: 10
Pattern: koznxbqjsx

Algorithm       Comparisons       Time (ms)       Matches
-----------------------------------------------------------------
Naive          10401             1.5191          1
Rabin-Karp     108               3.0369          1
KMP            10385             1.2769          1

Pattern Length: 20
Pattern: biedoonfykhjtfzzutfs

Algorithm       Comparisons       Time (ms)       Matches
-----------------------------------------------------------------
Naive    